# Defining goods, bads and intermediates

## Step 1 - Assigning the defintions

| Classification | Definition over the 12-month performance window | Binary target (`target_bad`) |
|---|---|---|
| Good | No missed payments in any of the twelve months. | 0 |
| Bad | 3+ MIA over the 12 month In-sample period | 1 |
| Intermediate | At least one missed-payment month, but less than 3 MIA for the In-Sample Period . | 0 |

### Plan for using intermediates
- I will be removing these from the model to run a known good bad two stage regression, this ill include delphi, then complementing this by addressing the residuals with applicant characteristics.
- Once this is done, I will score and apply this score to the indeterminate population, which I can then use to decide on a cut-off, then applying this to a KIGB model. 


In [2]:
# Locating the data and notebooks

from pathlib import Path
import pandas as pd
from IPython.display import display

# Find the project root from either the root or notebooks folder.
current_folder = Path.cwd()

if (current_folder / "data").is_dir():
    PROJECT_DIR = current_folder
elif (current_folder.parent / "data").is_dir():
    PROJECT_DIR = current_folder.parent
else:
    raise FileNotFoundError(
        "Could not locate a data folder in the current or parent directory."
    )

DATA_DIR = PROJECT_DIR / "data"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"

working_data = pd.read_csv(DATA_DIR / "working data.csv")

print(f"Data folder: {DATA_DIR}")
print(f"Notebooks folder: {NOTEBOOKS_DIR}")
print(f"Accounts loaded: {len(working_data):,}")

Data folder: c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data
Notebooks folder: c:\Users\georg\OneDrive\Coding\Origination-Scorecard\notebooks
Accounts loaded: 9,913


In [3]:
# Checking the required payment columns exist

payment_columns = [
    f"missed_payment_m{month:02d}"
    for month in range(1, 13)
]

required_columns = ["account_id", *payment_columns]

missing_columns = [
    column for column in required_columns
    if column not in working_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("All required columns are present.")

All required columns are present.


In [4]:
# Checking for any missing monthly payment data - removing any accounts with missing data

# Convert monthly payment values to numeric.
# Missing or nonnumeric values become NaN.
payments = working_data[payment_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

# Flag accounts:
# 1 = at least one missing or invalid monthly observation
# 0 = all twelve monthly observations are valid (0 or 1)
working_data["invalid_payment_data"] = (
    ~payments.isin([0, 1]).all(axis=1)
).astype(int)

# Review the flag before excluding accounts.
display(
    working_data["invalid_payment_data"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("invalid_payment_data")
    .reset_index(name="accounts")
)

# Copy flagged accounts into the exceptions dataset.
exceptions = working_data.loc[
    working_data["invalid_payment_data"].eq(1)
].copy()

exceptions["exception_reason"] = (
    "Missing or invalid monthly performance values"
)

# Export exceptions, preserving their original monthly values.
exceptions.to_csv(
    DATA_DIR / "exceptions.csv",
    index=False,
)

# Retain accounts with twelve valid monthly observations.
working_data = working_data.loc[
    working_data["invalid_payment_data"].eq(0)
].copy()

# Align validated payment values with the retained accounts.
payments = payments.loc[working_data.index].astype(int)
working_data[payment_columns] = payments

print(f"Exceptions exported: {len(exceptions):,}")
print(f"Accounts retained: {len(working_data):,}")

,invalid_payment_data,accounts
0,0,9913
1,1,0


Exceptions exported: 0
Accounts retained: 9,913


Positive result, previous DQ checks have avoided including any rows with missing datapoints over the 12 month performance

In [5]:
# Count the missing payments by account ID

if working_data["account_id"].isna().any():
    raise ValueError("Missing account_id values found.")

if working_data["account_id"].duplicated().any():
    raise ValueError(
        "Duplicate account_id values found. Resolve these before "
        "counting missed payments over the twelve-month window."
    )

working_data["missed_payment_months"] = (
    payments.sum(axis=1).astype(int)
)

In [6]:
# Apply new descriptive labels to the accounts based on the number of missed payments

is_good = working_data["missed_payment_months"].eq(0)
is_bad = working_data["missed_payment_months"].ge(3)

working_data["outcome"] = "Indeterminate"
working_data.loc[is_good, "outcome"] = "Known Good"
working_data.loc[is_bad, "outcome"] = "Known Bad"

In [7]:
# Create Binary Variable for Known Good vs. Known Bad

# Known Bad = 1; Known Good and Indeterminate = 0.
working_data["target_bad"] = (
    working_data["outcome"].eq("Known Bad").astype(int)
)

In [8]:
# Show population breakdown

classification_order = [
    "Known Good",
    "Known Bad",
    "Indeterminate",
]

definitions = {
    "Known Good": "0 missed-payment months",
    "Known Bad": "3–12 missed-payment months",
    "Indeterminate": "1–2 missed-payment months",
}

breakdown = (
    working_data["outcome"]
    .value_counts()
    .reindex(classification_order, fill_value=0)
    .rename_axis("classification")
    .reset_index(name="accounts")
)

breakdown["definition"] = breakdown["classification"].map(definitions)

total_accounts = breakdown["accounts"].sum()

breakdown["percentage"] = (
    breakdown["accounts"].div(total_accounts).mul(100)
    if total_accounts
    else 0.0
)

display(
    breakdown[
        ["classification", "definition", "accounts", "percentage"]
    ].style.format({
        "accounts": "{:,.0f}",
        "percentage": "{:.2f}%",
    })
)

,classification,definition,accounts,percentage
0,Known Good,0 missed-payment months,"9,320",94.02%
1,Known Bad,3–12 missed-payment months,396,3.99%
2,Indeterminate,1–2 missed-payment months,197,1.99%


In [9]:
# Remove Indetermindates and store seperately for later use in the model

is_indeterminate = working_data["outcome"].eq("Indeterminate")

indeterminates = working_data.loc[is_indeterminate].copy()

indeterminates.to_csv(
    DATA_DIR / "indeterminates.csv",
    index=False,
)

print(f"Indeterminates exported: {len(indeterminates):,}")

Indeterminates exported: 197


This now leaves us with a KGB model

In [10]:
# Known Good Bad model dataset

working_data = working_data.loc[
    working_data["outcome"].isin(["Known Good", "Known Bad"])
].copy()

print(f"Accounts retained for modelling: {len(working_data):,}")

Accounts retained for modelling: 9,716


In [11]:
# Review retained dataset

display(
    working_data[
        [
            "account_id",
            *payment_columns,
            "missed_payment_months",
            "outcome",
            "target_bad",
        ]
    ].head(10)
)

display(
    working_data.groupby(["outcome", "target_bad"])
    .size()
    .rename("accounts")
    .reset_index()
)

,account_id,missed_payment_m01,missed_payment_m02,missed_payment_m03,missed_payment_m04,missed_payment_m05,missed_payment_m06,missed_payment_m07,missed_payment_m08,missed_payment_m09,missed_payment_m10,missed_payment_m11,missed_payment_m12,missed_payment_months,outcome,target_bad
0,ACC-10000000,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
1,ACC-10000001,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
2,ACC-10000002,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
3,ACC-10000003,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
4,ACC-10000004,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
5,ACC-10000005,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
6,ACC-10000006,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
7,ACC-10000007,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
8,ACC-10000008,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0
9,ACC-10000009,0,0,0,0,0,0,0,0,0,0,0,0,0,Known Good,0


,outcome,target_bad,accounts
0,Known Bad,1,396
1,Known Good,0,9320


In [12]:
# Exporting the KGB dataset

# Export the retained Known Good and Known Bad modelling dataset.
output_path = DATA_DIR / "KGB model.csv"

working_data.to_csv(output_path, index=False)

print(f"Exported {len(working_data):,} accounts to {output_path}")

Exported 9,716 accounts to c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB model.csv
